<a href="https://colab.research.google.com/github/AdityaJare/ML-PRojects/blob/main/mental_health_llm(bert).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
pip install datasets

In [4]:
from transformers import pipeline
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from torch.utils.data import dataloader
from datasets import Dataset
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


In [5]:
data = pd.read_csv("/content/Combined Data mental health llm.csv")
data.dropna(inplace=True)
data.drop(columns=['Unnamed: 0'], axis=1, inplace=True)

In [6]:
data

,statement,status
0,oh my gosh,Anxiety
1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,I've shifted my focus to something else but I'...,Anxiety
4,"I'm restless and restless, it's been a month n...",Anxiety
...,...,...
53038,Nobody takes me seriously I’ve (24M) dealt wit...,Anxiety
53039,"selfishness ""I don't feel very good, it's lik...",Anxiety
53040,Is there any way to sleep better? I can't slee...,Anxiety
53041,"Public speaking tips? Hi, all. I have to give ...",Anxiety


In [7]:
data["status"].value_counts()

,count
status,
Normal,16343
Depression,15404
Suicidal,10652
Anxiety,3841
Bipolar,2777
Stress,2587
Personality disorder,1077


In [8]:
data = data.sample(n=6000, random_state=42).reset_index(drop=True)

In [9]:
data

,statement,status
0,I'm lazy to complain about it ba ihh,Normal
1,i think the wifi on my iphone is broken it wil...,Normal
2,Good tracking apps? I've been trying to find a...,Bipolar
3,I have recently looked into reddit and found t...,Depression
4,that's your favorite thing to do?,Normal
...,...,...
5995,What is around you right now? A photo? A comfo...,Depression
5996,I am tired. I am ready for everything to be ov...,Suicidal
5997,? What if I am at a friends party and a man wh...,Stress
5998,if it add any kind of info m almost the thing ...,Depression


In [10]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stopword = set(stopwords.words("english"))

def clean_statement(text):
  text = text.lower()

  #remove everything expect letters
  text = re.sub(r"[^a-zA-Z\s]", '', text)

  #tokenization
  words = text.split()
  print(words)
  words = [word for word in words if word not in stopword]


  return " ".join(words)

data["statement"] = data["statement"].apply(clean_statement)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Streaming output truncated to the last 5000 lines.
['ill', 'be', 'celebrating', 'eid', 'soon', 'if', 'i', 'make', 'a', 'mistake', 'ill', 'ask', 'for', 'thr', 'okay']
['my', 'prayer', 'today', 'god', 'dont', 'let', 'my', 'love', 'for', 'you', 'be', 'replaced', 'by', 'my', 'love', 'for', 'your', 'creatures', 'when', 'someone', 'who', 'means', 'the', 'most', 'has', 'gone', 'do', 'not', 'remain', 'sad']
['i', 'have', 'been', 'feeling', 'really', 'slow', 'recently', 'i', 'do', 'eat', 'a', 'lot', 'of', 'meat', 'but', 'twice', 'a', 'week', 'i', 'drink', 'alcohol', 'and', 'eat', 'little', 'which', 'affects', 'zinc', 'i', 'also', 'have', 'auto', 'immune', 'in', 'past', 'my', 'zinc', 'was', 'ok', 'but', 'at', 'that', 'point', 'i', 'believe', 'i', 'was', 'taking', 'mg', 'a', 'day', 'so', 'perhaps', 'without', 'that', 'i', 'could', 'be', 'low', 'anyway', 'have', 'not', 'had', 'much', 'for', 'a', 'while', 'and', 'i', 'have', 'a', 'large', 'spot', 'on', 'my', 'back', 'and', 'tiny', 'spot', 'on', 'fa

In [11]:
data

,statement,status
0,im lazy complain ba ihh,Normal
1,think wifi iphone broken quot connect quot act...,Normal
2,good tracking apps ive trying find apps track ...,Bipolar
3,recently looked reddit found place actually qu...,Depression
4,thats favorite thing,Normal
...,...,...
5995,around right photo comfortable place sit beaut...,Depression
5996,tired ready everything life pointless painful ...,Suicidal
5997,friends party man whispers ear likes stare tou...,Stress
5998,add kind info almost thing ive always felt kin...,Depression


In [12]:
data["status"].value_counts()

,count
status,
Normal,1894
Depression,1730
Suicidal,1219
Anxiety,429
Stress,304
Bipolar,293
Personality disorder,131


In [13]:
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(sampling_strategy = "auto", random_state = 42)

x = data.drop("status", axis=1)
y = data["status"]

x_resampled, y_resampled = ros.fit_resample(x, y)

data = pd.concat([x_resampled, y_resampled], axis = 1)

print(data["status"].value_counts())

status
Normal                  1894
Bipolar                 1894
Depression              1894
Suicidal                1894
Stress                  1894
Personality disorder    1894
Anxiety                 1894
Name: count, dtype: int64


In [14]:
le = LabelEncoder()

data["label"] = le.fit_transform(data["status"])

data

,statement,status,label
0,im lazy complain ba ihh,Normal,3
1,think wifi iphone broken quot connect quot act...,Normal,3
2,good tracking apps ive trying find apps track ...,Bipolar,1
3,recently looked reddit found place actually qu...,Depression,2
4,thats favorite thing,Normal,3
...,...,...,...
13253,cannot afford therapist social anxiety depress...,Suicidal,6
13254,would like die simplei would really like suffe...,Suicidal,6
13255,death day time set affairs might even get bedr...,Suicidal,6
13256,hard seeing others live life would kill know w...,Suicidal,6


In [15]:
x_train,x_test, y_train, y_test = train_test_split(data["statement"], data["label"], test_size = 0.2, random_state=42)

In [16]:
tokenizer =  AutoTokenizer.from_pretrained('bert-base-uncased')

train_encoding=tokenizer(list(x_train), padding = True, truncation = True, max_length = 200)
test_encoding=tokenizer(list(x_test), padding = True, truncation = True, max_length = 200)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [17]:
train_dataset = Dataset.from_dict({"input_ids" : train_encoding["input_ids"], 'attention_mask':train_encoding["attention_mask"], "labels":y_train.tolist()})
test_dataset = Dataset.from_dict({"input_ids" : test_encoding["input_ids"], 'attention_mask':test_encoding["attention_mask"], "labels":y_test.tolist()})

In [18]:
print(train_dataset)
print(train_dataset[1])

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 10606
})
{'input_ids': [101, 10657, 2360, 2651, 2428, 7823, 3728, 2288, 5086, 3435, 2833, 4825, 2651, 2353, 2154, 2731, 10365, 2183, 2477, 2288, 3255, 4218, 3244, 6085, 2371, 2428, 4629, 8726, 2318, 16058, 6634, 6834, 5191, 2242, 3308, 6069, 6752, 2047, 3105, 2134, 2102, 2215, 2681, 2220, 2191, 3087, 4737, 2134, 2102, 2215, 3280, 5723, 4385, 2371, 6634, 2886, 3225, 3108, 2371, 3082, 2371, 6015, 6427, 2242, 3308, 8948, 2144, 2524, 7200, 2092, 3266, 2562, 6634, 2886, 3724, 5670, 2130, 2295, 2288, 2919, 14978, 2197, 3178, 2288, 2188, 9069, 10047, 2145, 6015, 2242, 3308, 8948, 2740, 10089, 11757, 15095, 2075, 5223, 12447, 2296, 2154, 15366, 5996, 2524, 2147, 6263, 11897, 3105, 2428, 14841, 4892, 2514, 2066, 6933, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [19]:
model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(le.classes_))

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
training_arg = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none"
)

In [21]:
trainer = Trainer(
    model=model,
    args= training_arg,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.355671,0.360719
2,0.186379,0.267689
3,0.191353,0.288151
4,0.080483,0.323655
5,0.064546,0.347959


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3315, training_loss=0.26528971731258194, metrics={'train_runtime': 2117.1656, 'train_samples_per_second': 25.048, 'train_steps_per_second': 1.566, 'total_flos': 5450549081100000.0, 'train_loss': 0.26528971731258194, 'epoch': 5.0})

In [22]:
results = trainer.evaluate()
print(results)

Training Loss,Validation Loss,Epoch
0.064546,0.347959,5


{'eval_loss': 0.34795886278152466}


In [23]:
trainer.save_model("./my_bert_model")
tokenizer.save_pretrained("./my_bert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_bert_model/tokenizer_config.json', './my_bert_model/tokenizer.json')